# 02 · Parcellation: source resolution, then the 18-region (r18) stores (CPU)

1. **Anatomy of the r18 atlas** (no EEG): each hemisphere's 34 Desikan-Killiany labels partitioned into 9 regions,
   no cross-hemisphere region, identical left / right definitions, every region one contiguous piece of cortex,
   correct hemispheric placement, non-empty source sets.
2. **Analytic source resolution** per atlas (68 / 34 / 18 / 12 regions), both inverses, with the stores' fsaverage
   BEM and inverse settings: leadfield rank, forced redundancy, cross-talk, own-region dominance, leakage.
3. **The stores' own operators** (real data, no labels): every development participant's saved 68-ROI operator
   (LCMV from the participant's own data covariance; wMNE) against the stores' cached forward model; feature
   reliability; whether coarse signals could be rebuilt from saved intermediates (they are not: §4 rebuilds).
4. **r18 stores** (approved by the author, 2026-10-18, conditional on steps 1-2): built from the ORIGINAL recordings
   by the unchanged store pipeline (merged Desikan-Killiany labels, `mean_flip` over their own sources; never averages
   of 68-ROI signals or PLV), LCMV and wMNE, window modes B and A; validated, wMNE matched to LCMV, and each matched
   window by window to its 68-ROI store (participants, labels, windows, split, sampling, sensor features).
5. **QC** of the four r18 stores, then the resolution of the r18 operators actually built and their reliability.

The 68-ROI stores are only read. r18 is NOT made the analysis atlas here: that is recorded by the author after
notebook 04. A rank-18 sensor space does not prove that 18 regions are independently or accurately localized; the
cross-talk and dominance numbers quantify how far they are. Resumable (participant checkpoints). CPU, about 2-3 h
in total (store builds ≈ 40 min LCMV + 30-45 min wMNE).

**How to use:** *Runtime → Run all*. Allow Google Drive access when asked. If Colab disconnects, open the notebook
again and *Run all* again: finished work is kept on Google Drive and skipped. Everything is written to
`<outputs>/FINAL_v1/` on Drive (never to the temporary Colab disk); the validated graph stores and all earlier
results are only read.

In [ ]:
# ===== Setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_CODE = False   # leave False. True only to move the release code pin deliberately to the newest commit
                      # (<outputs>/FINAL_v1/CODE_COMMIT.txt; unfinished work is then recomputed).
BRANCH = 'claude/unified-final-framework'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py'] + (['--update-code'] if UPDATE_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
FINAL = OUT + '/FINAL_v1'
print('release outputs on Drive:', FINAL)

In [ ]:
import glob, json
import pandas as pd
from IPython.display import Image, display
pd.set_option('display.max_rows', 400, 'display.width', 250, 'display.max_columns', 40)

def latest(pattern):
    found = sorted(glob.glob(pattern))
    return found[-1] if found else None
import json

## 1. Anatomical checks of the r18 atlas

In [ ]:
ATLAS_OK = True
try:
    sh("python scripts/final.py atlas-check --atlas r18")
except RuntimeError:
    ATLAS_OK = False

## 2. Analytic source resolution and the atlas recommendation (stores' BEM)

In [ ]:
sh("python scripts/final.py parcellation")

In [ ]:
d = latest(FINAL + '/parcellation/resolution_*')
display(pd.read_csv(d + '/resolution_by_atlas.csv').round(3))
rec = json.load(open(d + '/recommendation.json'))
print('recommended atlas:', rec['recommended'], '|', rec['forward_model'])
for v in rec['verdicts']:
    print(v['atlas'], 'eligible' if v['eligible'] else 'not eligible', '; '.join(v['reasons']))
R18_ELIGIBLE = any(v['atlas'] == 'r18' and v['eligible'] for v in rec['verdicts'])
if 'approx' in rec['forward_model'] or 'sphere' in rec['forward_model']:
    print("NOTE: the stores' BEM was not found; this is an approximate head model (docs/final/parcellation.md)")

## 3. The 68-ROI stores' own operators, feature reliability, reconstruction check (read-only)

In [ ]:
for inverse in ('lcmv', 'wmne'):
    sh(f"python scripts/final.py store-resolution --inverse {inverse} --atlas dk68")
    sh(f"python scripts/final.py reliability --inverse {inverse} --mode B")
    sh(f"python scripts/final.py reconstruct-check --inverse {inverse} --mode B")

## 4. r18 stores from the original recordings (approved; runs only if steps 1-2 passed)

Each step is resumable: completed participants are skipped, and `validation.json`, `match.json` and
`atlas_match.json` are written once. New directories next to the 68-ROI stores (outside `FINAL_v1/`):
`D3_LCMV_XU/graphsLX-r18_v1_<hash>`, `graphsLX10-r18_v1_<hash>`, `D3_WMNE_XU/graphsWX-r18_v1_<hash>`,
`graphsWX10-r18_v1_<hash>`; their sizes are printed below.

In [ ]:
BUILD_R18 = True   # approved by the author (2026-10-18), conditional on the checks above
if not (ATLAS_OK and R18_ELIGIBLE):
    raise RuntimeError(f"r18 not built: atlas checks passed = {ATLAS_OK}, r18 meets the resolution rule with the "
                       f"stores' BEM = {R18_ELIGIBLE}. The 68-ROI stores remain the analysis; report to the author.")
if BUILD_R18:
    for inverse in ('lcmv', 'wmne'):
        sh(f"python scripts/build_stores.py build --inverse {inverse} --atlas r18")
        sh(f"python scripts/build_stores.py validate --inverse {inverse} --atlas r18")
    sh("python scripts/build_stores.py match --atlas r18")
    for inverse in ('lcmv', 'wmne'):
        sh(f"python scripts/build_stores.py match-atlas --inverse {inverse} --atlas r18")
        sh(f"python scripts/build_stores.py size --inverse {inverse} --atlas r18")
    sh("python scripts/final.py stores --atlas r18")      # verification of the four r18 stores (locked when QC opens them)

## 5. QC of the r18 stores, then the resolution and reliability of the r18 operators actually built

In [ ]:
ATLAS = 'r18'
UNRESOLVED = []
for inverse in ('lcmv', 'wmne'):
    for mode in ('B', 'A'):
        try:
            sh(f"python scripts/final.py qc --inverse {inverse} --mode {mode} --atlas {ATLAS}")
        except RuntimeError:
            UNRESOLVED.append((inverse, mode))
from IPython.display import Markdown
for d in sorted(glob.glob(FINAL + f'/qc/*_*_{ATLAS}/')):
    if os.path.exists(d + 'qc_report.md'):
        display(Markdown(open(d + 'qc_report.md').read()))
if UNRESOLVED:
    raise RuntimeError(f'category A findings need a decision for {ATLAS} {UNRESOLVED}: see the reports above and '
                       'docs/final/qc.md; experiments refuse to start until then')

In [ ]:
for inverse in ('lcmv', 'wmne'):
    sh(f"python scripts/final.py store-resolution --inverse {inverse} --atlas r18")
    sh(f"python scripts/final.py reliability --inverse {inverse} --mode B --atlas r18")
for inverse in ('lcmv', 'wmne'):
    for atlas in ('dk68', 'r18'):
        d = latest(f"{FINAL}/parcellation/store_resolution_{inverse}_{atlas}_*")
        t = pd.read_csv(d + '/per_participant.csv')
        print(inverse, atlas, t[['operator_rank', 'redundant_signals', 'crosstalk_fraction_median',
                                 'rois_dominated_by_own_region', 'leakage_corr_abs_median']].median().round(3).to_dict())